# Seed-Budget Study (Patched & Stable)
**What this notebook does**
- Per-class split: **70% train pool / 20% val / 10% test** (real-only val/test).
- For each seed budget **S**, we take **S real seeds per class from the train pool**, then augment to a fixed target per class **for training only**.
- **ResNet18** with ImageNet normalization, conservative optimizer (AdamW), cosine schedule, early stopping, AMP **off** by default for stability with tiny S.
- Optional class-balanced loss (effective number) with safe clamping.
- Saves per-run history, summaries, and confusion matrices.


In [ ]:

# ===== Imports & basic config =====
import os, random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import datasets as tvds, transforms as T, models
from torchvision.transforms import InterpolationMode

from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt

import albumentations as A
import cv2

DATA_ROOT = r"/home/sanghee/wafer_project/WM-811K_ImageFolder"
# ---- paths & high-level settings ----
RESULTS_DIR = Path("./seed_budget_results_5 runs3"); RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- split counts (per class) ----
VAL_PER_CLASS, TEST_PER_CLASS = 30, 15

# ---- training & augmentation targets ----
IMG_SIZE = (64, 64)          # slightly larger helps small defects
TOTAL_PER_CLASS = 4000       # lower than 10k to avoid heavy repetition for small S
S_LIST = [2, 5, 10, 20, 40, 80, 100]   # adjust as needed

# ---- training hyperparameters ----
EPOCHS = 12
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
USE_AMP = False              # keep off for tiny-S stability; turn on later if desired
FREEZE_EPOCHS = 2            # freeze backbone first few epochs to stabilize learning
BATCH_SIZE = 64
NUM_WORKERS = 0              # Windows-safe
SEED = 42

# ---- class options ----
DONT_AUGMENT_NAMES = {"unknown", "none"}  # set() to augment all classes

# ---- device & seeds ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Device:", DEVICE)


# ===== Build per-class split once (train_pool / val / test) =====
base = tvds.ImageFolder(DATA_ROOT)
CLASSES = base.classes
NUM_CLASSES = len(CLASSES)
print("CLASSES:", CLASSES)

dont_aug_ids = {i for i,c in enumerate(CLASSES) if c.lower() in DONT_AUGMENT_NAMES}

paths_per_class = defaultdict(list)
for p, y in base.samples:
    paths_per_class[y].append(p)

rng = np.random.RandomState(SEED + 111)
splits = {}
rows = []
for c_id, cname in enumerate(CLASSES):
    paths = paths_per_class[c_id]
    n = len(paths)
    if n < 3:
        raise RuntimeError(f"Class '{cname}' has only {n} images; need >=3 to split.")
    idx = np.arange(n); rng.shuffle(idx)
    # Use fixed counts per class (cap to available samples)
    n_val  = min(VAL_PER_CLASS, n)
    n_test = min(TEST_PER_CLASS, max(0, n - n_val))
    n_train = max(0, n - n_val - n_test)
    # shuffled indices
    idx = np.arange(n); rng.shuffle(idx)
    train_idx = idx[:n_train]; val_idx = idx[n_train:n_train+n_val]; test_idx = idx[n_train+n_val:n_train+n_val+n_test]
    splits[c_id] = {
        "train_pool": [paths[i] for i in train_idx],
        "val": [paths[i] for i in val_idx],
        "test":[paths[i] for i in test_idx],
    }
    rows.append({"class": cname, "train_pool": len(splits[c_id]["train_pool"]),
                 "val": len(splits[c_id]["val"]), "test": len(splits[c_id]["test"])})
print(pd.DataFrame(rows))

# ===== Transforms & dataset helpers (torchvision-only, matches Hybrid_Train) =====
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_aug(img_size):
    # Wafer-safe, moderate; SAME style as Hybrid base_aug, plus Normalize
    return T.Compose([
        T.Resize(img_size),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomRotation((0, 90), expand=False),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def eval_tfm(img_size):
    return T.Compose([
        T.Resize(img_size),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

aug_train = build_aug(IMG_SIZE)
tfm_eval  = eval_tfm(IMG_SIZE)

class AlbFromPaths(Dataset):
    """
    TRAIN dataset from (path,label) pairs.
    NOTE: Now torchvision-only. 'aug' must be a torchvision Compose taking a PIL image.
    """
    def __init__(self, file_label_list, aug):
        self.data = file_label_list
        self.aug  = aug  # torchvision transforms.Compose
    def __len__(self): 
        return len(self.data)
    def __getitem__(self, i):
        p, y = self.data[i]
        img = Image.open(p).convert("RGB")
        img = self.aug(img)   # torchvision call (no 'image=')
        return img, y

class EvalFromPaths(Dataset):
    """VAL/TEST dataset (real-only)."""
    def __init__(self, file_label_list):
        self.data = file_label_list
        self.tfm  = tfm_eval
    def __len__(self): 
        return len(self.data)
    def __getitem__(self, i):
        p, y = self.data[i]
        img = Image.open(p).convert("RGB")
        img = self.tfm(img)   # torchvision call
        return img, y

class ProxySynthetic(Dataset):
    """
    Synthetic generator from seed paths (train-only).
    Torchvision-only augmentation for consistency with Hybrid_Train.
    """
    def __init__(self, seed_paths, label, aug, target_len, base_seed=123, class_id=0):
        assert len(seed_paths) > 0
        self.seed_paths = list(seed_paths)
        self.label = int(label)
        self.aug   = aug      # torchvision transforms.Compose
        self.target_len = int(target_len)
        self.rs    = np.random.RandomState(base_seed + 97*int(class_id))
    def __len__(self): 
        return self.target_len
    def __getitem__(self, i):
        p = self.seed_paths[self.rs.randint(0, len(self.seed_paths))]
        # (Optional) make stochastic ops reproducible per-sample by reseeding Python/NumPy here if you like
        img = Image.open(p).convert("RGB")
        img = self.aug(img)   # torchvision call
        return img, self.label

def seed_worker(worker_id):
    import numpy as _np, random as _random
    base_seed = torch.initial_seed() % 2**32
    _np.random.seed(base_seed + worker_id)
    _random.seed(base_seed + worker_id)


# ===== Builders for a given seed budget S =====
def build_seed_budget_datasets(S, base_seed=SEED):
    train_parts, val_parts, test_parts = [], [], []
    for c_id, cname in enumerate(CLASSES):
        tp = splits[c_id]["train_pool"]
        vp = splits[c_id]["val"]
        xp = splits[c_id]["test"]
        if len(tp) == 0:
            raise RuntimeError(f"Class '{cname}' has empty train_pool after split.")
        S_eff = min(S, len(tp))
        seeds = tp[:S_eff]  # deterministic; change to np.random.choice for random per run

        # TRAIN
        train_real = [(p, c_id) for p in seeds]
        if c_id in dont_aug_ids:
            train_ds_c = AlbFromPaths(train_real, aug_train)
        else:
            need = max(0, TOTAL_PER_CLASS - len(train_real))
            synth = ProxySynthetic(seeds if len(seeds)>0 else [tp[0]],
                                   c_id, aug_train, need, base_seed=base_seed, class_id=c_id)
            train_ds_c = ConcatDataset([AlbFromPaths(train_real, aug_train), synth]) if need>0 else AlbFromPaths(train_real, aug_train)

        # VAL/TEST (real-only)
        val_ds_c  = EvalFromPaths([(p, c_id) for p in vp])
        test_ds_c = EvalFromPaths([(p, c_id) for p in xp])

        train_parts.append(train_ds_c); val_parts.append(val_ds_c); test_parts.append(test_ds_c)

    train_ds = ConcatDataset(train_parts)
    val_ds   = ConcatDataset(val_parts)
    test_ds  = ConcatDataset(test_parts)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                              worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(base_seed))
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    return train_loader, val_loader, test_loader


# ===== Class weights (effective number) =====
def compute_label_hist_from_loader(loader, num_batches=50):
    cnt = Counter()
    for i, (_, y) in enumerate(loader):
        cnt.update(map(int, y.tolist()))
        if i+1 >= num_batches: break
    return cnt


def class_weights_effective_num(cnt, num_classes, beta=0.999, clamp=(0.3, 3.0), device=None):
    w = []
    for c in range(num_classes):
        n = max(1, int(cnt.get(c, 0)))
        eff = (1.0 - (beta ** n)) / (1.0 - beta)
        wc = 1.0 / max(eff, 1e-8)
        w.append(wc)
    w = np.array(w, dtype=np.float64)
    w = w / w.mean()
    w = np.clip(w, clamp[0], clamp[1])
    t = torch.tensor(w, dtype=torch.float32)
    return t.to(device) if device is not None else t


# ===== Model, evaluation, and training loop (stable) =====
def make_resnet18(num_classes, pretrained=True, dropout=0.2):
    if pretrained:
        try:
            weights = models.ResNet18_Weights.IMAGENET1K_V1
        except AttributeError:
            weights = "IMAGENET1K_V1"
        net = models.resnet18(weights=weights)
    else:
        net = models.resnet18(weights=None)
    in_feats = net.fc.in_features
    net.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feats, num_classes)) if dropout else nn.Linear(in_feats, num_classes)
    return net.to(DEVICE)

@torch.no_grad()
def evaluate_model(model, loader, tag, save=True):
    model.eval()
    ys, yh = [], []
    for x,y in loader:
        x = x.to(DEVICE); out = model(x)
        ys.extend(y.tolist()); yh.extend(out.argmax(1).cpu().tolist())
    ys = np.array(ys); yh = np.array(yh)
    cm = confusion_matrix(ys, yh, labels=list(range(NUM_CLASSES)))
    per_class_acc = np.diag(cm) / np.clip(cm.sum(1), 1, None)
    prec, rec, f1, sup = precision_recall_fscore_support(ys, yh, labels=list(range(NUM_CLASSES)), zero_division=0)
    overall = float((ys==yh).mean())

    if save:
        (RESULTS_DIR / "runs").mkdir(exist_ok=True)
        pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(RESULTS_DIR / f"{tag}_cm.csv")
        pd.DataFrame({"class": CLASSES, "acc": per_class_acc, "precision": prec, "recall": rec, "f1": f1, "support": sup}).to_csv(RESULTS_DIR / f"{tag}_per_class.csv", index=False)
        # PNG
        plt.figure(figsize=(8,6)); plt.imshow(cm, interpolation='nearest', aspect='auto')
        plt.title(f"Confusion Matrix ({tag})"); plt.colorbar()
        ticks = np.arange(len(CLASSES)); plt.xticks(ticks, CLASSES, rotation=45, ha='right'); plt.yticks(ticks, CLASSES)
        thresh = cm.max()/2 if cm.size else 0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                v = int(cm[i,j]); plt.text(j, i, str(v), ha='center', va='center', color='white' if v>thresh else 'black', fontsize=8)
        plt.ylabel('True'); plt.xlabel('Pred'); plt.tight_layout(); plt.savefig(RESULTS_DIR / f"{tag}_cm.png", dpi=150); plt.close()
    return {"overall_acc": overall, "per_class_acc": per_class_acc, "precision": prec, "recall": rec, "f1": f1, "support": sup}

def fit_and_report(model, train_loader, val_loader, test_loader, tag, weights=None,
                   epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY, label_smooth=LABEL_SMOOTH,
                   use_amp=USE_AMP, freeze_epochs=FREEZE_EPOCHS):
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smooth)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler() if (use_amp and torch.cuda.is_available()) else None

    # freeze backbone for first few epochs
    def set_backbone_requires_grad(flag: bool):
        for n,p in model.named_parameters():
            if n.startswith("fc."):
                p.requires_grad = True
            else:
                p.requires_grad = flag
    if freeze_epochs > 0:
        set_backbone_requires_grad(False)

    best_val, best_state, bad, patience = -1.0, None, 0, 3
    history_rows = []

    for ep in range(1, epochs+1):
        if freeze_epochs > 0 and ep == (freeze_epochs+1):
            set_backbone_requires_grad(True)

        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for x,y in train_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    out = model(x); loss = criterion(out, y)
                if not torch.isfinite(loss):
                    print("Non-finite loss; skipping batch."); continue
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                out = model(x); loss = criterion(out, y)
                if not torch.isfinite(loss):
                    print("Non-finite loss; skipping batch."); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            loss_sum += float(loss.item())
            pred = out.argmax(1); total += y.size(0); correct += (pred==y).sum().item()

        train_loss = loss_sum / max(1, len(train_loader)); train_acc = correct / max(1, total)
        val_metrics = evaluate_model(model, val_loader, tag=f"{tag}_val_tmp", save=False)
        history_rows.append({"epoch": ep, "train_loss": train_loss, "train_acc": train_acc, "val_acc": val_metrics["overall_acc"]})
        print(f"Epoch {ep:02d}/{epochs}  train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  val_acc={val_metrics['overall_acc']:.3f}")

        scheduler.step()
        curr = val_metrics["overall_acc"]
        if curr > best_val + 1e-4:
            best_val, bad = curr, 0
            best_state = {k: v.detach().cpu() for k,v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"Early stopping at epoch {ep} (best val={best_val:.3f})")
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k,v in best_state.items()})

    val_final = evaluate_model(model, val_loader, tag=f"{tag}_VAL", save=True)
    test_final= evaluate_model(model, test_loader, tag=f"{tag}_TEST", save=True)

    pd.DataFrame(history_rows).to_csv(RESULTS_DIR / f"{tag}_history.csv", index=False)
    pd.DataFrame([{"val_acc": val_final["overall_acc"], "test_acc": test_final["overall_acc"]}]).to_csv(RESULTS_DIR / f"{tag}_summary.csv", index=False)
    return {"val": val_final, "test": test_final}

# ===== Multi-repeat seed-budget sweep (collects variance + per-class F1) =====
REPEATS = 5                     # increase for smoother error bars
#S_LIST  = [2, 4, 8, 16, 32, 64] # seed budgets to test

all_rows = []
per_class_rows = []

for S in S_LIST:
    for r in range(REPEATS):
        base_seed = SEED + 1000*r + S
        print(f"\n=== S={S}  repeat={r} (base_seed={base_seed}) ===")

        # 1) data
        train_loader, val_loader, test_loader = build_seed_budget_datasets(S, base_seed=base_seed)

        # 2) model + balanced loss
        model = make_resnet18(NUM_CLASSES, pretrained=True, dropout=0.2)
        cnt = compute_label_hist_from_loader(train_loader, num_batches=50)
        #weights = class_weights_effective_num(cnt, NUM_CLASSES, beta=0.999, clamp=(0.3, 3.0), device=DEVICE)

        # 3) train/eval
        tag = f"S{S}_r{r}"
        out = fit_and_report(
            model, train_loader, val_loader, test_loader,
            tag=tag, weights=None,
            epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY,
            label_smooth=LABEL_SMOOTH,
            use_amp=USE_AMP, freeze_epochs=FREEZE_EPOCHS
        )

        # 4) record macro & per-class
        macro_f1 = float(np.mean(out["test"]["f1"]))
        all_rows.append({
            "S": S,
            "repeat": r,
            "macro_f1": macro_f1,
            "val_acc": float(out["val"]["overall_acc"]),
            "test_acc": float(out["test"]["overall_acc"]),
        })
        for i, cls in enumerate(CLASSES):
            per_class_rows.append({
                "S": S,
                "repeat": r,
                "class": cls,
                "f1": float(out["test"]["f1"][i]),
                "precision": float(out["test"]["precision"][i]),
                "recall": float(out["test"]["recall"][i]),
                "acc": float(out["test"]["per_class_acc"][i]),
                "support": int(out["test"]["support"][i]),
            })

# Save raw tables
results_df    = pd.DataFrame(all_rows)
per_class_df  = pd.DataFrame(per_class_rows)
results_df.to_csv(RESULTS_DIR / "seed_budget_repeat_summary.csv", index=False)
per_class_df.to_csv(RESULTS_DIR / "seed_budget_per_class_repeat.csv", index=False)
print("Saved:", RESULTS_DIR / "seed_budget_repeat_summary.csv")
print("Saved:", RESULTS_DIR / "seed_budget_per_class_repeat.csv")


In [ ]:
# ===== Plot A: Macro-F1 mean ± SD vs S =====
g = results_df.groupby("S")["macro_f1"].agg(["mean","std"]).reset_index()

plt.figure(figsize=(6,4))
plt.errorbar(g["S"], g["mean"], yerr=g["std"], marker="o", capsize=4)
#plt.xscale("log", base=2)
plt.ylim(0.4, 1.0)
plt.xlabel("Seeds per class (S)")
plt.ylabel("Macro F1 (mean over 5 repeats)")
#plt.title("Seed–Budget Performance (mean ± 1 SD)")
plt.grid(True, alpha=0.3)
plt.show()

# ===== Plot B: Per-class F1 vs S (mean across repeats) =====
pc = per_class_df.groupby(["class","S"])["f1"].mean().reset_index()

plt.figure(figsize=(6,4))
for cls in CLASSES:
    sub = pc[pc["class"] == cls]
    if len(sub)==0: 
        continue
    plt.plot(sub["S"], sub["f1"], marker="o", label=cls)

#plt.xscale("log", base=2)
plt.ylim(0.0, 1.0)
plt.yticks(np.arange(0, 1.01, 0.1))
plt.xlabel("Seeds per class (S)")
plt.ylabel("Per-class F1 (mean over 5 repeats)")
#plt.title("Per-class F1 vs. Seed Budget (mean over repeats)")
plt.legend(ncol=min(len(CLASSES), 5), fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# (Optional) Per-class error bands (mean ± SD)
pc_stats = per_class_df.groupby(["class","S"])["f1"].agg(["mean","std"]).reset_index()

plt.figure(figsize=(6,4))
for cls in CLASSES:
    sub = pc_stats[pc_stats["class"] == cls]
    if len(sub)==0: 
        continue
    x = sub["S"].values
    m = sub["mean"].values
    s = sub["std"].values
    plt.plot(x, m, marker="o", linewidth=1.5, label=cls)
    # light band for ±1 SD
    plt.fill_between(x, np.clip(m - s, 0, 1), np.clip(m + s, 0, 1), alpha=0.10)

#plt.xscale("log", base=2)
plt.ylim(0, 1.0)
plt.yticks(np.arange(0, 1.01, 0.1))
plt.xlabel("Seeds per class (S)")
plt.ylabel("Per-class F1 (mean ± SD)")
#plt.title("Per-class F1 vs. Seed Budget (with variability)")
plt.legend(ncol=min(len(CLASSES), 5), fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
